In [ ]:
### HERE COMES THE ENITRE ANALYSIS AND THE PLOTS I WILL CREATE
### IT WILL ACCESS THE SAME FUNCTIONS (PARTIALLY) AS THE 4_homologs_analysis_visualization.ipynb

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path for custom src imports later
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Default plot style
# sns.set_theme(style="whitegrid", context="notebook")
# plt.rcParams["figure.dpi"] = 100

In [ ]:
DATA_DIR = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"

df = pd.read_parquet(f"{DATA_DIR}/variants_annotated_final.parquet")
print(f"Loaded {len(df):,} variant-region assignments")
print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")

# Also load the regions JSON — useful for context (WT sequences, etc.)
with open(f"{DATA_DIR}/genomic_coords_merged_win5.json") as f:
    regions = json.load(f)
region_by_id = {r["region_id"]: r for r in regions}
print(f"Loaded {len(regions)} regions from JSON")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MANUAL FIX: coordinate convention inconsistency in regions JSON
# TODO: fix at source in 16_ev_signature_predictor/src/write_bed_file.py
# or wherever genomic_coords_merged_win5.json gets generated.
#
# Issue: most regions use inclusive-inclusive (len = end - start + 1),
# but 12 regions use half-open (len = end - start). 6 regions are broken
# (seq_len 0 or 1). This patches them for analysis only.
# ═══════════════════════════════════════════════════════════════════════════

# Inspect the half-open regions
halfopen_region_ids = []
broken_region_ids = []
for r in regions:
    start, end = r["prot_region"]
    seq_len = len(r["prot_seq"])
    if seq_len < 2:
        broken_region_ids.append(r["region_id"])
    elif seq_len == end - start:
        halfopen_region_ids.append(r["region_id"])

print(f"Half-open regions to patch: {len(halfopen_region_ids)}")
for rid in halfopen_region_ids:
    r = region_by_id[rid]
    print(f"  {rid}: prot_region={r['prot_region']}, "
          f"seq='{r['prot_seq']}' (len {len(r['prot_seq'])})")

print(f"\nBroken regions to drop: {len(broken_region_ids)}")
for rid in broken_region_ids:
    r = region_by_id[rid]
    print(f"  {rid}: prot_region={r['prot_region']}, seq_len={len(r['prot_seq'])}")


# ═══════════════════════════════════════════════════════════════════════════
# MANUAL FIX: apply the patch
# ═══════════════════════════════════════════════════════════════════════════

# 1. Patch half-open regions — shift to 1-based inclusive
for rid in halfopen_region_ids:
    r = region_by_id[rid]
    old_start, old_end = r["prot_region"]
    # Half-open [0, N) -> inclusive-inclusive 1-based [1, N]
    r["prot_region"] = [old_start + 1, old_end]
    # Sanity check
    assert len(r["prot_seq"]) == r["prot_region"][1] - r["prot_region"][0] + 1, \
        f"Patch failed for {rid}"

# 2. Drop broken regions
for rid in broken_region_ids:
    del region_by_id[rid]

# 3. Flag affected variants for traceability
df["coord_patched"] = df["region_id"].isin(halfopen_region_ids)
df["coord_dropped"] = df["region_id"].isin(broken_region_ids)
print(f"Variants in patched regions: {df['coord_patched'].sum()}")
print(f"Variants in dropped regions: {df['coord_dropped'].sum()}")

# 4. Remove variants in dropped regions from df
df = df[~df["coord_dropped"]].copy()
print(f"Remaining variants after drop: {len(df):,}")

# 5. Update region_start_aa in df for patched regions
patched_mask = df["region_id"].isin(halfopen_region_ids)
df.loc[patched_mask, "region_start_aa"] = df.loc[patched_mask, "region_start_aa"] + 1

# Also update region_end_aa stays the same (only start changed)
# (end was already consistent with the actual sequence end)

In [ ]:
def _check(row):
    region = region_by_id.get(row["region_id"])
    if region is None or pd.isna(row["protein_position_int"]):
        return None
    pos = int(row["protein_position_int"]) - int(row["region_start_aa"])
    if pos < 0 or pos >= len(region["prot_seq"]):
        return None
    if row["before_aa"] is None:
        return None
    # Skip insertions (before_aa is '-') and multi-residue cases
    if row["before_aa"] == "-" or len(row["before_aa"]) != 1:
        return None
    return region["prot_seq"][pos] == row["before_aa"]

df["wt_match"] = df.apply(_check, axis=1)
print(df["wt_match"].value_counts(dropna=False))

In [ ]:
# Flag reliability for RG-level analysis
df["rg_analysis_reliable"] = df["wt_match"] == True

# Any analysis requiring exact region sequence should subset on this
df_for_rg = df[df["rg_analysis_reliable"]].copy()

print(f"Total variants: {len(df):,}")
print(f"Usable for RG analysis: {len(df_for_rg):,}")
print(f"\nBreakdown by group:")
print(df_for_rg["group"].value_counts())

In [ ]:
###### NOW ANALYSIS

In [ ]:
import src.analysis_visualization.region_analysis as ra

fig, stats = ra.plot_variant_density(df_for_rg, dataset="gnomad")
plt.show()

In [ ]:
fig, results = ra.plot_consequence_distributions(df_for_rg, dataset="gnomad")
plt.show()

In [ ]:
fig, r = ra.plot_median_alphamissense(df_for_rg, dataset="gnomad")
plt.show()

In [ ]:
import src.analysis_visualization.rg_analysis as rga
df_rg = rga.compute_rg_disruption_columns(df_for_rg, region_by_id)
print("RG hit columns added. Summary:")
print(df_rg["hits_rg"].value_counts())

# Run each plot independently
fig_a, r_a = rga.plot_region_length(region_by_id, dataset="gnomad")
plt.show()

fig_b, r_b = rga.plot_n_rg_motifs(region_by_id, dataset="gnomad")
plt.show()

fig_c, r_c = rga.plot_rg_density(region_by_id, dataset="gnomad")
plt.show()

# D1 — density version (replaces the saturating D)
fig_d1, r_d1 = rga.plot_variants_per_rg_by_type(df_rg, region_by_id, dataset="gnomad")
plt.show()

# # D2 — AlphaMissense on RG-hitting missense
# fig_d2, r_d2 = rga.plot_median_alphamissense_on_rgs(df_rg, dataset="gnomad")
# plt.show()


In [ ]:
df_rg

In [ ]:
fig, r = rga.plot_rg_role_asymmetry(df_rg, dataset="gnomad")
plt.show()

In [ ]:
results = rga.plot_rg_change_events_stacked(df_rg, region_by_id, dataset="gnomad")

# the single event version has been commmented out in the rg-analysis, because it showed no signifincance in any plot, so here is just the stacked version

In [ ]:
# Make sure df_events is computed (already have compute_rg_change_events from before)
df_events = rga.compute_rg_change_events(df_rg, region_by_id)

# Analysis 1
loss_transition_results = rga.plot_rg_loss_transitions(df_events, dataset="gnomad")
plt.show()

# Analysis 2
cluster_results = rga.plot_isolated_vs_clustered_loss(
    df_events, region_by_id,
    window_sizes=[2,4,6],
    dataset="gnomad",
)
plt.show()

In [ ]:
gain_transition_results = rga.plot_rg_gain_transitions(df_events, dataset="gnomad")
plt.show()

In [ ]:
# # Build the null once — reused for both plots
# null_results = rga.build_enumeration_null(region_by_id, df_for_rg)

# # RG events observed vs expected
# rg_comparison = rga.plot_rg_events_observed_vs_expected(
#     df_events, null_results, dataset="gnomad",
# )
# plt.show()

# # Consequences observed vs expected
# cons_comparison = rga.plot_consequences_observed_vs_expected(
#     df_for_rg, null_results, dataset="gnomad",
# )
# plt.show()

In [ ]:
# Build null once
null_results = rga.build_enumeration_null(region_by_id, df_for_rg)

# RG events: 4 subplots (no_change / loss / gain / movement)
rg_results = rga.plot_rg_events_vs_expected_boxes(
    df_events, null_results, dataset="gnomad",
)
plt.show()

# Consequences: 3 subplots (synonymous / missense / nonsense)
cons_results = rga.plot_consequences_vs_expected_boxes(
    df_for_rg, null_results, dataset="gnomad",
)
plt.show()

In [ ]:
# Plot A: per-variant distribution, R/G-affecting only
fig_a, r_a = rga.plot_delta_rg_ratio_per_variant(df_rg, region_by_id, dataset="gnomad")
plt.show()

# Plot B: per-region mean across all missense
fig_b, r_b = rga.plot_delta_rg_ratio_per_region(df_rg, region_by_id, dataset="gnomad")
plt.show()

In [ ]:
############### PHYSCHEM

In [ ]:
import src.analysis_visualization.physchem_analysis as pca

# # Step 1: compute per-variant deltas (slow — uses multiprocessing)
# deltas_df = pca.compute_physchem_deltas(df_rg, region_by_id)

# # Save the result — expensive to recompute
# deltas_df.to_parquet(
#     "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/physchem_deltas.parquet"
# )

deltas_df = pd.read_parquet("/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/physchem_deltas.parquet"
)

# Step 2: aggregate to per-region means for classifier features
per_region = pca.aggregate_per_region(deltas_df)

# Step 3: plot each feature individually
all_results = pca.plot_all_delta_features(per_region, dataset="gnomad")

# Bonus: per-region WT baseline features (no variants involved — useful for classifier too)
wt_features = pca.compute_wt_physchem(region_by_id)

In [ ]:
################# AMINO ACID SUBSTITUTIONS

In [ ]:
import src.analysis_visualization.substitution_matrix_analysis as smx

# All-missense matrix (single call)
enrichment = smx.run_substitution_analysis(
    df_rg, dataset="gnomad", min_total=5,
)

# Access individual outputs
log2_or_matrix = enrichment["log2_or"]
fdr_matrix = enrichment["fdr"]
counts_pos = enrichment["counts_pos"]

In [ ]:
import src.analysis_visualization.substitution_matrix_analysis as smx

# Side-by-side rare (AF < 1e-4) vs common (AF >= 1e-3) matrices
results = smx.plot_af_comparison_matrices(
    df_rg,
    af_rare_max=1e-5,
    af_common_min=1e-5,
    dataset="gnomad",
)

# Access individual results
rare_enrichment = results["rare"]
common_enrichment = results["common"]

In [ ]:
# Total G→X variants in common matrix, per group
r_common = results["common"]
g_row_pos = r_common["counts_pos"].loc["G"].sum()
g_row_neg = r_common["counts_neg"].loc["G"].sum()
print(f"G→anything: pos = {g_row_pos}, neg = {g_row_neg}")
print(f"G→S fraction: pos = {77/g_row_pos:.3f}, neg = {19/g_row_neg:.3f}")

In [ ]:
# Quick check
gs_pos = df_rg[
    (df_rg["before_aa"] == "G") & 
    (df_rg["after_aa"] == "S") & 
    (df_rg["group"] == "pos") &
    (df_rg["AF_joint"] >= 1e-5) &
    (df_rg["Consequence"].str.contains("missense_variant", na=False))
]
gs_neg = df_rg[
    (df_rg["before_aa"] == "G") & 
    (df_rg["after_aa"] == "S") & 
    (df_rg["group"] == "neg") &
    (df_rg["AF_joint"] >= 1e-5) &
    (df_rg["Consequence"].str.contains("missense_variant", na=False))
]
print("G→S codon distribution in pos:")
print(gs_pos["Codons"].value_counts().head(10))
print("\nG→S codon distribution in neg:")
print(gs_neg["Codons"].value_counts().head(10))

In [ ]:
import src.analysis_visualization.codon_usage as cu

results = cu.run_codon_usage_analysis(region_by_id, dataset="gnomad")

# # For your G→S specific question:
# print("\nGlycine codon proportions:")
# print(f"  Pos:  GGC={results['proportions']['pos']['G']['GGC']:.3f}, "
#       f"GGT={results['proportions']['pos']['G']['GGT']:.3f}, "
#       f"GGA={results['proportions']['pos']['G']['GGA']:.3f}, "
#       f"GGG={results['proportions']['pos']['G']['GGG']:.3f}")
# print(f"  Neg:  GGC={results['proportions']['neg']['G']['GGC']:.3f}, "
#       f"GGT={results['proportions']['neg']['G']['GGT']:.3f}, "
#       f"GGA={results['proportions']['neg']['G']['GGA']:.3f}, "
#       f"GGG={results['proportions']['neg']['G']['GGG']:.3f}")
# print(f"  Ref:  GGC={results['reference_proportions']['G']['GGC']:.3f}, "
#       f"GGT={results['reference_proportions']['G']['GGT']:.3f}, "
#       f"GGA={results['reference_proportions']['G']['GGA']:.3f}, "
#       f"GGG={results['reference_proportions']['G']['GGG']:.3f}")

In [ ]:
df_rg

In [ ]:
import src.analysis_visualization.substitution_matrix_analysis as smx
# Full pipeline — observed + expected + enrichment + plot
result = smx.run_composition_normalized_analysis(
    df_rg,
    region_by_id,
    min_total=5,
    dataset="gnomad",
)

In [ ]:
print("Expected pos matrix (non-zero cells):")
exp_pos = result["exp_pos"]
nonzero_exp = exp_pos[exp_pos > 0]
print(f"  Total cells with expected > 0: {(exp_pos > 0).sum().sum()} / 400")
print(f"  Expected total (should match observed total): {exp_pos.values.sum():.1f}")
print(f"  Observed pos total: {result['obs_pos'].values.sum()}")

print("\nRatios — sample non-NaN pos ratios:")
ratio_pos = result["ratio_pos"]
finite_pos = ratio_pos[ratio_pos.notna()].stack()
print(f"  Cells with finite ratio: {len(finite_pos)} / 400")
print(f"  Ratio summary: median={finite_pos.median():.3f}, "
      f"min={finite_pos.min():.3f}, max={finite_pos.max():.3f}")
print(f"  First 10 values:")
print(finite_pos.head(10))

In [ ]:
 
result = smx.run_codon_substitution_analysis(
    df_rg
)

# def run_codon_substitution_analysis(
#     df: pd.DataFrame,
#     group_col: str = "group",
#     pos_label: str = "pos",
#     neg_label: str = "neg",
#     min_total: int = 5,
#     dataset: str = "gnomad",
#     save: bool = True,
#     **plot_kwargs,
# ) -> dict:
#     """
#     [Dataset-agnostic]
#     End-to-end codon substitution matrix: build counts, enrichment, plot.
#     """
#     df_pos = df[df[group_col] == pos_label]
#     df_neg = df[df[group_col] == neg_label]
 
#     counts_pos = compute_codon_substitution_counts(df_pos)
#     counts_neg = compute_codon_substitution_counts(df_neg)
 
#     enrichment = compute_codon_enrichment(counts_pos, counts_neg, min_total=min_total)
#     plot_codon_substitution_matrix(enrichment, dataset=dataset, save=save, **plot_kwargs)
 
#     return enrichment
 

In [ ]:
#### AF analysis

In [ ]:
import src.analysis_visualization.af_spectrum as afs

stats = afs.plot_af_spectrum_cdf(df_rg, dataset="gnomad")

In [ ]:
import src.analysis_visualization.af_spectrum as afs

# df_rg must have `is_rg_disrupting` column (from compute_rg_disruption_columns)
stats = afs.plot_af_spectrum_by_subset(df_rg, dataset="gnomad")

In [ ]:
# Quick check in notebook
rg = df_rg[df_rg["is_rg_disrupting"] & df_rg["AF_joint"].notna()]
for group in ["pos", "neg"]:
    g = rg[rg["group"] == group]["AF_joint"]
    common = g[g >= 1e-4]
    very_common = g[g >= 1e-3]
    print(f"{group}: n={len(g):,}, n≥1e-4={len(common)}, n≥1e-3={len(very_common)}, "
          f"max={g.max():.4f}")

In [ ]:
#$###### CLASSIFIER FEATURE COLLECTION

In [ ]:
import src.analysis_visualization.classifier_features as cf

# If you have physchem_deltas computed, load it; otherwise pass None
try:
    physchem_deltas_df = pd.read_parquet(
        "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/physchem_deltas.parquet"
    )
except FileNotFoundError:
    physchem_deltas_df = None

features_df = cf.build_classifier_features(
    df_rg=df_rg,
    df_events=df_events,
    region_by_id=region_by_id,
    physchem_deltas_df=physchem_deltas_df,
    dataset="gnomad",
)

# Save it
cf.save_features(features_df)

In [ ]:
features_df

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Quick RF classifier test: with AM only / without AM / with all features
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, roc_curve,
)
from sklearn.impute import SimpleImputer

# ── Setup ────────────────────────────────────────────────────────────────
RANDOM_STATE = 42
N_TREES = 100

EXCLUDE_COLS = {"region_id", "group", "label", "group_x", "group_y"}
AM_FEATURES = [c for c in features_df.columns
               if "alphamissense" in c.lower() or c.startswith("am_")
               or c == "fraction_pathogenic"]

print(f"Total features: {len(features_df.columns) - len(EXCLUDE_COLS)}")
print(f"AlphaMissense features ({len(AM_FEATURES)}): {AM_FEATURES}")
print(f"Dataset: {len(features_df)} regions "
      f"({(features_df['group'] == 'pos').sum()} pos, "
      f"{(features_df['group'] == 'neg').sum()} neg)")

y = (features_df["group"] == "pos").astype(int).values

def _run_rf(feature_subset, label):
    # Keep only numeric features
    X_df = features_df[feature_subset].select_dtypes(include=[np.number])
    dropped = set(feature_subset) - set(X_df.columns)
    if dropped:
        print(f"  (Dropped {len(dropped)} non-numeric features: {sorted(dropped)})")
    feature_subset = list(X_df.columns)
    X = X_df.values

    imputer = SimpleImputer(strategy="median")
    X_imputed = imputer.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_imputed, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
    )
    rf = RandomForestClassifier(
        n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    y_proba = rf.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_auc = cross_val_score(
        RandomForestClassifier(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1),
        X_imputed, y, scoring="roc_auc", cv=cv,
    )
    cv_acc = cross_val_score(
        RandomForestClassifier(n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1),
        X_imputed, y, scoring="accuracy", cv=cv,
    )

    rf_full = RandomForestClassifier(
        n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1,
    )
    rf_full.fit(X_imputed, y)
    importances = pd.Series(
        rf_full.feature_importances_, index=feature_subset,
    ).sort_values(ascending=False)

    print(f"\n── {label} ────────────────────────────────────")
    print(f"  Features: {len(feature_subset)}")
    print(f"  Test set (80/20): accuracy = {acc:.3f}, AUC = {auc:.3f}")
    print(f"  5-fold CV:        accuracy = {cv_acc.mean():.3f} ± {cv_acc.std():.3f}, "
          f"AUC = {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")
    print(f"\n  Top 15 features by importance:")
    print(importances.head(15).to_string())

    return {
        "label": label,
        "test_accuracy": acc,
        "test_auc": auc,
        "cv_accuracy_mean": cv_acc.mean(),
        "cv_accuracy_std": cv_acc.std(),
        "cv_auc_mean": cv_auc.mean(),
        "cv_auc_std": cv_auc.std(),
        "importances": importances,
        "y_test": y_test,
        "y_proba": y_proba,
    }
# ── Run three configurations ────────────────────────────────────────────
all_features = [c for c in features_df.columns if c not in EXCLUDE_COLS]
features_without_am = [c for c in all_features if c not in AM_FEATURES]
features_only_am = list(AM_FEATURES)

result_all = _run_rf(all_features, "ALL features (with AM)")
result_without_am = _run_rf(features_without_am, "WITHOUT AlphaMissense")
result_only_am = _run_rf(features_only_am, "AM ONLY")


# ── Comparative visualization ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: ROC curves for all three configurations
ax = axes[0]
styles = [("-", "tab:blue"), ("--", "tab:green"), (":", "tab:orange")]
for r, (linestyle, color) in zip(
    [result_all, result_without_am, result_only_am], styles,
):
    fpr, tpr, _ = roc_curve(r["y_test"], r["y_proba"])
    ax.plot(fpr, tpr, linestyle=linestyle, color=color, linewidth=1.5,
            label=f"{r['label']} (AUC={r['test_auc']:.3f})")
ax.plot([0, 1], [0, 1], "k:", linewidth=0.6, alpha=0.5)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("ROC comparison")
ax.legend(frameon=False, fontsize=9, loc="lower right")
ax.grid(alpha=0.3, linestyle=":", linewidth=0.4)

# Right: Top-20 feature importances (WITH AM)
ax = axes[1]
top20 = result_all["importances"].head(20)
colors = ["tab:orange" if f in AM_FEATURES else "tab:blue" for f in top20.index]
ax.barh(range(len(top20)), top20.values[::-1],
        color=colors[::-1], edgecolor="black", linewidth=0.3)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index[::-1], fontsize=7)
ax.set_xlabel("Feature importance")
ax.set_title("Top 20 features (ALL)  — orange = AM")
ax.grid(axis="x", alpha=0.3, linestyle=":", linewidth=0.4)

for side in ("top", "right"):
    axes[0].spines[side].set_visible(False)
    axes[1].spines[side].set_visible(False)

plt.tight_layout()
plt.show()

# ── Summary ─────────────────────────────────────────────────────────────
print("\n" + "═" * 60)
print("SUMMARY — 5-fold CV")
print("═" * 60)
print(f"{'Configuration':<30} {'CV AUC':>15} {'CV accuracy':>18}")
print("─" * 65)
for r in [result_all, result_without_am, result_only_am]:
    auc_str = f"{r['cv_auc_mean']:.3f} ± {r['cv_auc_std']:.3f}"
    acc_str = f"{r['cv_accuracy_mean']:.3f} ± {r['cv_accuracy_std']:.3f}"
    print(f"{r['label']:<30} {auc_str:>15} {acc_str:>18}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Leave-one-gene-out CV: honest generalization test
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42
N_TREES = 100
EXCLUDE_COLS = {"region_id", "group", "label", "group_x", "group_y"}
AM_FEATURES = [c for c in features_df.columns
               if "alphamissense" in c.lower() or c.startswith("am_")]

# Extract gene (UniProt accession) from region_id
# region_id format: UniProtAccession_start_end
features_df = features_df.copy()
features_df["gene"] = features_df["region_id"].str.split("_").str[0]

n_genes = features_df["gene"].nunique()
region_per_gene = features_df["gene"].value_counts()
print(f"Number of unique genes: {n_genes}")
print(f"Regions per gene: min={region_per_gene.min()}, "
      f"median={region_per_gene.median():.0f}, "
      f"max={region_per_gene.max()}, "
      f"mean={region_per_gene.mean():.2f}")
print(f"Genes with >1 region: {(region_per_gene > 1).sum()}")

y = (features_df["group"] == "pos").astype(int).values
groups = features_df["gene"].values

def _run_logo(feature_subset, label):
    # Keep only numeric
    X_df = features_df[feature_subset].select_dtypes(include=[np.number])
    feature_subset = list(X_df.columns)
    X = X_df.values

    # Impute
    imputer = SimpleImputer(strategy="median")
    X_imputed = imputer.fit_transform(X)

    # Leave-one-gene-out CV
    logo = LeaveOneGroupOut()

    # Get per-region predictions via out-of-fold prediction
    rf = RandomForestClassifier(
        n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1,
    )
    y_proba = cross_val_predict(
        rf, X_imputed, y, groups=groups, cv=logo, method="predict_proba",
        n_jobs=-1,
    )[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    auc = roc_auc_score(y, y_proba)
    acc = accuracy_score(y, y_pred)

    print(f"\n── {label} ────────────────────────────────────")
    print(f"  Features: {len(feature_subset)}")
    print(f"  LOGO-CV accuracy: {acc:.3f}")
    print(f"  LOGO-CV AUC:      {auc:.3f}")

    return {
        "label": label,
        "auc": auc,
        "accuracy": acc,
        "y_true": y,
        "y_proba": y_proba,
    }


all_features = [c for c in features_df.columns
                if c not in EXCLUDE_COLS and c != "gene"]
features_without_am = [c for c in all_features if c not in AM_FEATURES]
features_only_am = list(AM_FEATURES)

result_all = _run_logo(all_features, "ALL features (with AM)")
result_without_am = _run_logo(features_without_am, "WITHOUT AlphaMissense")
result_only_am = _run_logo(features_only_am, "AM ONLY")


# ── ROC comparison figure ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
styles = [("-", "tab:blue"), ("--", "tab:green"), (":", "tab:orange")]
for r, (ls, color) in zip(
    [result_all, result_without_am, result_only_am], styles,
):
    fpr, tpr, _ = roc_curve(r["y_true"], r["y_proba"])
    ax.plot(fpr, tpr, linestyle=ls, color=color, linewidth=1.5,
            label=f"{r['label']} (AUC={r['auc']:.3f})")
ax.plot([0, 1], [0, 1], "k:", linewidth=0.6, alpha=0.5)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("Leave-one-gene-out ROC")
ax.legend(frameon=False, fontsize=9, loc="lower right")
ax.grid(alpha=0.3, linestyle=":", linewidth=0.4)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()


# ── Final comparison vs earlier stratified 5-fold ──────────────────────
print("\n" + "═" * 60)
print("HONEST SUMMARY: stratified 5-fold vs leave-one-gene-out")
print("═" * 60)
print(f"{'Configuration':<30} {'5-fold AUC':>15} {'LOGO AUC':>15}")
print("─" * 60)
print(f"{'ALL features (with AM)':<30} {'0.838':>15} {result_all['auc']:>15.3f}")
print(f"{'WITHOUT AlphaMissense':<30} {'0.797':>15} {result_without_am['auc']:>15.3f}")
print(f"{'AM ONLY':<30} {'0.786':>15} {result_only_am['auc']:>15.3f}")
print("\nIf LOGO numbers are MUCH lower than 5-fold, the classifier was")
print("largely learning gene identity, not generalizable biology.")

In [ ]:
# Right: Top-20 feature importances (WITH AM)
fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(1, 1, 1)
top20 = result_all["importances"].head(20)
colors = ["tab:orange" if f in AM_FEATURES else "tab:blue" for f in top20.index]
ax.barh(range(len(top20)), top20.values[::-1],
        color=colors[::-1], edgecolor="black", linewidth=0.3)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index[::-1], fontsize=7)
ax.set_xlabel("Feature importance")
ax.set_title("Top 20 features (ALL)  — orange = AM")
ax.grid(axis="x", alpha=0.3, linestyle=":", linewidth=0.4)

for side in ("top", "right"):
    axes[0].spines[side].set_visible(False)
    axes[1].spines[side].set_visible(False)

plt.tight_layout()
plt.show()